In [ ]:
# Merges ascii files downloaded from FDA on file type over years and quarters
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import os
import gc
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

BASE_PATH   = "/kaggle/input/datasets/maxthacker/fda-faers-ascii-dataset/ascii"
OUTPUT_PATH = "/kaggle/working/faers_merged"
FILE_TYPES  = ["demo", "drug", "indi", "outc", "reac", "rpsr", "ther"]
READ_KWARGS = dict(delimiter="$", encoding="latin-1", dtype=str)

os.makedirs(OUTPUT_PATH, exist_ok=True)

year_quarters = [
    (yr, q)
    for yr in range(2012, 2026)
    for q in range(1, 5)
    if not (yr == 2012 and q != 4)
]

SUPERSET_COLS = {
    "demo": ["primaryid","caseid","caseversion","i_f_code","event_dt","mfr_dt",
             "init_fda_dt","fda_dt","rept_cod","auth_num","mfr_num","mfr_sndr",
             "lit_ref","age","age_cod","age_grp","gndr_cod","sex","e_sub","wt",
             "wt_cod","rept_dt","to_mfr","occp_cod","reporter_country",
             "occr_country","source_year","source_quarter"],
    "drug": ["primaryid","caseid","drug_seq","role_cod","drugname","prod_ai",
             "val_vbm","route","dose_vbm","cum_dose_chr","cum_dose_unit",
             "dechal","rechal","lot_num","exp_dt","nda_num","dose_amt",
             "dose_unit","dose_form","dose_freq","source_year","source_quarter"],
    "indi": ["primaryid","caseid","indi_drug_seq","indi_pt",
             "source_year","source_quarter"],
    "outc": ["primaryid","outc_cod","source_year","source_quarter"],
    "reac": ["primaryid","caseid","pt","drug_rec_act","source_year","source_quarter"],
    "rpsr": ["primaryid","rpsr_cod","source_year","source_quarter"],
    "ther": ["primaryid","caseid","dsg_drug_seq","start_dt","end_dt","dur",
             "dur_cod","source_year","source_quarter"],
}

def find_file_ci(folder, target_name):
    if not os.path.isdir(folder):
        return None
    target_lower = target_name.lower()
    for fname in os.listdir(folder):
        if fname.lower() == target_lower:
            return os.path.join(folder, fname)
    return None

def read_one_file(fpath, yr, q, superset):
    """Read and align a single file — runs in a thread."""
    df = pd.read_csv(fpath, **READ_KWARGS)
    df.columns = df.columns.str.strip().str.lower()
    df["source_year"]    = str(yr)
    df["source_quarter"] = str(q)
    return df.reindex(columns=superset)

# ── Main loop ─────────────────────────────────────────────────────────────────
missing_files = []
N_THREADS = 4  

for ft in tqdm(FILE_TYPES, desc="File types", position=0):
    out_path   = os.path.join(OUTPUT_PATH, f"{ft}.parquet")
    superset   = SUPERSET_COLS[ft]
    pa_schema  = pa.schema([pa.field(c, pa.string()) for c in superset])
    writer     = pq.ParquetWriter(out_path, pa_schema, compression="snappy")
    total_rows = 0
    errors     = []

    # Build the full list of (fpath, yr, q) tasks for this file type
    tasks = []
    for yr, q in year_quarters:
        yr2    = str(yr)[2:]
        folder = os.path.join(BASE_PATH, f"{yr}_Q{q}")
        fpath  = find_file_ci(folder, f"{ft}{yr2}q{q}.txt")
        if fpath is None:
            missing_files.append(os.path.join(folder, f"{ft}{yr2}q{q}.txt"))
        else:
            tasks.append((fpath, yr, q))

    # Read N_THREADS files in parallel, write results as they complete
    with ThreadPoolExecutor(max_workers=N_THREADS) as executor:
        futures = {
            executor.submit(read_one_file, fpath, yr, q, superset): (fpath, yr, q)
            for fpath, yr, q in tasks
        }
        for future in tqdm(as_completed(futures), total=len(futures),
                           desc=f"  {ft:4s}", position=1, leave=False):
            fpath, yr, q = futures[future]
            try:
                df = future.result()
                writer.write_table(pa.Table.from_pandas(df, schema=pa_schema, preserve_index=False))
                total_rows += len(df)
                del df
            except Exception as e:
                errors.append(f"{os.path.basename(fpath)}: {e}")

    writer.close()
    tqdm.write(f"Done: [{ft.upper()}]  {total_rows:>12,} rows  {os.path.getsize(out_path)/1e6:.1f} MB")
    if errors:
        for err in errors: tqdm.write(f"{err}")

    gc.collect()

if missing_files:
    tqdm.write(f"\n {len(missing_files)} missing file(s) — inspect `missing_files`")

print("\n Done!")